In [1]:
from pathlib import Path
import pandas as pd
import json
import os

In [2]:
top_path = Path(os.path.dirname(os.getcwd()))
data_path = top_path / "data"
insights_path = top_path / "reports" / "insights"

notebooks_path = top_path / "notebooks"
team_data_path = data_path / "processed" / "team_data.parquet"
predictions_path = insights_path / "LightGBM_predictions.parquet"

In [3]:
predictions_df = pd.read_parquet(predictions_path).rename(columns={"result": "prediction"})
team_df = pd.read_parquet(team_data_path)

## Store the accuracy per league in a dictionary

In [4]:
league_data = team_df[["gameid", "side", "league", "result"]]
predictions_df = predictions_df.merge(league_data, on=["gameid", "side"])

In [5]:
# Explore accuracy by league
leagues = predictions_df["league"].unique()
accuracies = {}

for league in leagues:
    league_df = predictions_df[predictions_df["league"] == league]
    accuracy = league_df["prediction"] == league_df["result"]
    accuracies[league] = {"count": len(league_df), "accuracy": accuracy.mean()}

accuracies = {k: v for k, v in sorted(accuracies.items(), key=lambda item: item[1]["accuracy"], reverse=True)}

with open(insights_path / "LightGBM_league_accuracies.json", "w") as f:
    json.dump(accuracies, f)

# Specific League Predictions Analysis

In [6]:
# Specific League Analysis
analysis_league = "LEC"
games_data =  team_df[["date", "gameid", "teamname", "opponentteam", "side"]]

lec_df = predictions_df[predictions_df["league"] == analysis_league][["gameid", "side", "league", "prediction", "result"]]
lec_df = games_data.merge(lec_df, on=["gameid", "side"])

lec_df["correct"] = lec_df["prediction"] == lec_df["result"]
lec_df.sort_values(by="date", inplace=True)

lec_df

,date,gameid,teamname,opponentteam,side,league,prediction,result,correct
0,2022-01-14 16:19:00,ESPORTSTMNT04_2090326,MAD Lions,Team Vitality,Blue,LEC,1,1,True
1,2022-01-14 16:19:00,ESPORTSTMNT04_2090326,Team Vitality,MAD Lions,Red,LEC,0,0,True
2,2022-01-14 18:45:50,ESPORTSTMNT01_2701805,Excel Esports,G2 Esports,Blue,LEC,0,0,True
3,2022-01-14 18:45:50,ESPORTSTMNT01_2701805,G2 Esports,Excel Esports,Red,LEC,1,1,True
4,2022-01-14 19:14:41,ESPORTSTMNT04_2090354,Astralis,Misfits Gaming,Blue,LEC,0,0,True
...,...,...,...,...,...,...,...,...,...
369,2024-04-13 16:48:15,LOLTMNT05_32056,Team BDS,Fnatic,Red,LEC,1,0,False
370,2024-04-14 15:12:53,LOLTMNT05_31072,G2 Esports,Fnatic,Blue,LEC,1,0,False
371,2024-04-14 15:12:53,LOLTMNT05_31072,Fnatic,G2 Esports,Red,LEC,0,1,False
372,2024-04-14 16:47:02,LOLTMNT05_32104,Fnatic,G2 Esports,Blue,LEC,0,0,True
